# Applied Cryptography, Project 3

Group 47

**Natania Jeffery (nj2153)**

**Joe Raymond (cjr10071)**

**Shreyas Sankpal (sss10093)**

**Enoch Sanyaolu (mos7641)**

Code for Problems 1 and 2. Problem 1 is done by hand in the report and only checked here. Standard library only.

In [1]:
import math
import time

---
## Problem 1: Checking the hand calculations

Blocks are 4-bit strings (A = 0000 to P = 1111) and E(b1 b2 b3 b4) = (b2 b3 b1 b4).

In [2]:
LETTERS = "ABCDEFGHIJKLMNOP"


def to_bits(text):
    """Letters A-P -> list of 4-bit strings."""
    return [format(LETTERS.index(ch), "04b") for ch in text]


def to_letters(blocks):
    return "".join(LETTERS[int(b, 2)] for b in blocks)


def xor(a, b):
    return format(int(a, 2) ^ int(b, 2), "04b")


def E(b):
    """E(b1 b2 b3 b4) = (b2 b3 b1 b4)"""
    return b[1] + b[2] + b[0] + b[3]


def D(c):
    """Inverse of E: D(c1 c2 c3 c4) = (c3 c1 c2 c4)"""
    return c[2] + c[0] + c[1] + c[3]


def ecb_encrypt(m):
    return [E(x) for x in m]


def ecb_decrypt(c):
    return [D(y) for y in c]


def cbc_encrypt(m, iv):
    c = [iv]
    for x in m:
        c.append(E(xor(x, c[-1])))
    return c[1:]


def cbc_decrypt(c, iv):
    previous = [iv] + c[:-1]
    return [xor(D(y), p) for y, p in zip(c, previous)]


def ctr_keystream(ctr, n):
    """E(ctr), E(ctr + 1), ... The first block uses ctr itself, as in the class example."""
    return [E(format((int(ctr, 2) + i) % 16, "04b")) for i in range(n)]


def ctr_encrypt(m, ctr):
    return [xor(x, k) for x, k in zip(m, ctr_keystream(ctr, len(m)))]


ctr_decrypt = ctr_encrypt   # same keystream XOR, D is never used

CBC and CTR send the IV or counter as the first block, as in the class examples.

In [3]:
m = to_bits("FOO")
IV = CTR = "1010"

c_ecb, c_cbc, c_ctr = ecb_encrypt(m), cbc_encrypt(m, IV), ctr_encrypt(m, CTR)
results = [("ECB", c_ecb, ecb_decrypt(c_ecb)),
           ("CBC", [IV] + c_cbc, cbc_decrypt(c_cbc, IV)),
           ("CTR", [CTR] + c_ctr, ctr_decrypt(c_ctr, CTR))]

for mode, sent, back in results:
    blocks = " ".join(sent)
    print(f"{mode}  sent: {blocks:<20} = {to_letters(sent):<5} decrypted: {to_letters(back)}")

ECB  sent: 1001 1110 1110       = JOO   decrypted: FOO
CBC  sent: 1010 1111 0001 1111  = KPBP  decrypted: FOO
CTR  sent: 1010 0011 1001 0100  = KDJE  decrypted: FOO


---
## Problem 2: Meet-in-the-middle attack on the mini block cipher

### Task 1a: Mini block cipher

Building blocks from the S-AES lecture. The state n0 n1 n2 n3 is the matrix [[n0, n2], [n1, n3]], and arithmetic is in GF(2^4) mod x^4 + x + 1.

In [4]:
SBOX = [0x9, 0x4, 0xA, 0xB, 0xD, 0x1, 0x8, 0x5,
        0x6, 0x2, 0x0, 0x3, 0xC, 0xE, 0xF, 0x7]
INV_SBOX = [SBOX.index(x) for x in range(16)]


def gf_mult(a, b):
    """Multiply two nibbles in GF(2^4) mod x^4 + x + 1."""
    result = 0
    while b:
        if b & 1:
            result ^= a
        a <<= 1
        if a & 0x10:
            a ^= 0b10011
        b >>= 1
    return result


def to_nibbles(state):
    return [(state >> 12) & 0xF, (state >> 8) & 0xF, (state >> 4) & 0xF, state & 0xF]


def from_nibbles(n):
    return (n[0] << 12) | (n[1] << 8) | (n[2] << 4) | n[3]


def substitute(state):
    return from_nibbles([SBOX[x] for x in to_nibbles(state)])


def inv_substitute(state):
    return from_nibbles([INV_SBOX[x] for x in to_nibbles(state)])


def shift(state):
    """Swap the two nibbles of the bottom row. Doing it twice gives the state back."""
    n0, n1, n2, n3 = to_nibbles(state)
    return from_nibbles([n0, n3, n2, n1])


def mix(state):
    """Multiply each column by [[1, 4], [4, 1]]."""
    n0, n1, n2, n3 = to_nibbles(state)
    return from_nibbles([n0 ^ gf_mult(4, n1), gf_mult(4, n0) ^ n1,
                         n2 ^ gf_mult(4, n3), gf_mult(4, n2) ^ n3])


def inv_mix(state):
    """Multiply each column by the inverse matrix [[9, 2], [2, 9]]."""
    n0, n1, n2, n3 = to_nibbles(state)
    return from_nibbles([gf_mult(9, n0) ^ gf_mult(2, n1), gf_mult(2, n0) ^ gf_mult(9, n1),
                         gf_mult(9, n2) ^ gf_mult(2, n3), gf_mult(2, n2) ^ gf_mult(9, n3)])


def add_round_key(state, key):
    return state ^ key

Key expansion: K = W0 W1 gives K1 = W2 W3 and K2 = W4 W5.

In [5]:
RCON1, RCON2 = 0b10000000, 0b00110000


def g(w, rcon):
    """RotNib, SubNib, then XOR with the round constant (8-bit word)."""
    rotated = ((w << 4) | (w >> 4)) & 0xFF
    return ((SBOX[rotated >> 4] << 4) | SBOX[rotated & 0xF]) ^ rcon


def key_expansion(key):
    """16-bit key K -> round keys (K1, K2)."""
    w0, w1 = key >> 8, key & 0xFF
    w2 = w0 ^ g(w1, RCON1)
    w3 = w2 ^ w1
    w4 = w2 ^ g(w3, RCON2)
    w5 = w4 ^ w3
    return (w2 << 8) | w3, (w4 << 8) | w5

The two rounds, as in the pseudo-code. Shift is its own inverse.

In [6]:
def encrypt_round1(k1, state):
    return add_round_key(mix(shift(substitute(state))), k1)


def encrypt_round2(k2, state):
    return add_round_key(shift(substitute(state)), k2)


def decrypt_round2(k2, state):
    return inv_substitute(shift(add_round_key(state, k2)))


def decrypt_round1(k1, state):
    return inv_substitute(shift(inv_mix(add_round_key(state, k1))))


def encrypt(key, plaintext):
    k1, k2 = key_expansion(key)
    return encrypt_round2(k2, encrypt_round1(k1, plaintext))


def decrypt(key, ciphertext):
    k1, k2 = key_expansion(key)
    return decrypt_round1(k1, decrypt_round2(k2, ciphertext))

**Check 1.** S-AES is the mini cipher plus AddRoundKey(K) at the start, so it should reproduce the lecture example (K = 0101 1001 0111 1010, plaintext "ZC").

In [7]:
def bits(x):
    """16-bit value as four nibbles, e.g. 0101 1001 0111 1010."""
    return " ".join(format(x, "016b")[i:i + 4] for i in range(0, 16, 4))


def saes_encrypt(key, plaintext):
    """Full S-AES from the lecture: AddRoundKey(K) first, then the two rounds above."""
    k1, k2 = key_expansion(key)
    return encrypt_round2(k2, encrypt_round1(k1, add_round_key(plaintext, key)))


K_LECTURE = 0b0101_1001_0111_1010
P_LECTURE = int.from_bytes(b"ZC", "big")

k1, k2 = key_expansion(K_LECTURE)
print("K1 = W2 W3 :", bits(k1))
print("K2 = W4 W5 :", bits(k2))
print("S-AES(ZC)  :", bits(saes_encrypt(K_LECTURE, P_LECTURE)))

K1 = W2 W3 : 1101 1100 1010 0110
K2 = W4 W5 : 0110 1100 1100 1010
S-AES(ZC)  : 1010 1001 1111 0110


The same example through the mini cipher. X is the middle state the attack matches on.

In [8]:
x = encrypt_round1(k1, P_LECTURE)
c = encrypt_round2(k2, x)
p = decrypt(K_LECTURE, c)
print("P             :", bits(P_LECTURE))
print("X (round 1)   :", bits(x))
print("C (round 2)   :", bits(c))
print("decrypt(K, C) :", bits(p), "=", p.to_bytes(2, "big").decode())

P             : 0101 1010 0100 0011
X (round 1)   : 0110 0011 0111 0111
C (round 2)   : 1110 1001 1001 0001
decrypt(K, C) : 0101 1010 0100 0011 = ZC


**Check 2.** Decryption undoes encryption for all 2^16 blocks.

In [9]:
test_keys = [0x0000, K_LECTURE, 0xFFFF]
ok = all(decrypt(k, encrypt(k, p)) == p for k in test_keys for p in range(1 << 16))
print("decrypt(encrypt(P)) == P for all 65,536 plaintexts and all test keys:", ok)

decrypt(encrypt(P)) == P for all 65,536 plaintexts and all test keys: True


### Task 1b: Ten plaintext-ciphertext pairs

The key was picked at random once and fixed so the results are reproducible. The attack only gets `pairs`.

In [10]:
SECRET_KEY = 0x71D9

words = ["HI", "GO", "OK", "NO", "UP", "ME", "WE", "IT", "ON", "SO"]
plaintexts = [int.from_bytes(w.encode(), "big") for w in words]
pairs = [(p, encrypt(SECRET_KEY, p)) for p in plaintexts]

print(" #  P    P (hex)  C (hex)  C (bits)")
for i, (w, (p, c)) in enumerate(zip(words, pairs), 1):
    print(f"{i:>2}  {w}   {p:04x}     {c:04x}     {bits(c)}")

 #  P    P (hex)  C (hex)  C (bits)
 1  HI   4849     02ec     0000 0010 1110 1100
 2  GO   474f     1c62     0001 1100 0110 0010
 3  OK   4f4b     afce     1010 1111 1100 1110
 4  NO   4e4f     10b2     0001 0000 1011 0010
 5  UP   5550     9d7a     1001 1101 0111 1010
 6  ME   4d45     b4f7     1011 0100 1111 0111
 7  WE   5745     0c60     0000 1100 0110 0000
 8  IT   4954     2f4b     0010 1111 0100 1011
 9  ON   4f4e     4fc4     0100 1111 1100 0100
10  SO   534f     da3c     1101 1010 0011 1100


### Task 2a: Meet-in-the-middle attack

Steps a to c: store X = encrypt_round1(K1, P) for every K1, then look up X' = decrypt_round2(K2, C) for every K2. Step d: drop key pairs that fail the next pairs. K1 and K2 are searched independently, without the key expansion.

In [11]:
def mitm(P, C):
    """Steps a to c: every (K1, K2) with encrypt_round1(K1, P) == decrypt_round2(K2, C)."""
    table = {}
    for k1 in range(1 << 16):
        table.setdefault(encrypt_round1(k1, P), []).append(k1)   # a
    matches = []
    for k2 in range(1 << 16):
        x = decrypt_round2(k2, C)                                # b
        matches += [(k1, k2) for k1 in table.get(x, [])]         # c
    return matches


def mitm_attack(pairs):
    """Step d: keep the matches of the first pair that also fit all the other pairs."""
    candidates = mitm(*pairs[0])
    counts = [len(candidates)]
    for P, C in pairs[1:]:
        candidates = [(k1, k2) for k1, k2 in candidates
                      if encrypt_round1(k1, P) == decrypt_round2(k2, C)]
        counts.append(len(candidates))
    return candidates, counts

### Task 2b: Results

In [12]:
start = time.perf_counter()
key_pairs, counts = mitm_attack(pairs)
print(f"attack time: {time.perf_counter() - start:.2f} s\n")

print("pairs used  (K1, K2) left")
for n, count in enumerate(counts, 1):
    print(f"{n:>10}  {count:>12,}")
print("\nkey pairs left:", [f"K1 = {k1:04x}, K2 = {k2:04x}" for k1, k2 in key_pairs])

attack time: 0.52 s

pairs used  (K1, K2) left
         1        65,536
         2            16
         3             1
         4             1
         5             1
         6             1
         7             1
         8             1
         9             1
        10             1

key pairs left: ['K1 = df06, K2 = 6660']


One pair leaves 2^16 key pairs, two leave 16 and three leave one.

**Step e.** Running the key expansion backwards from K1 gives K. Then any ciphertext can be decrypted, here a new message.

In [13]:
def invert_key_expansion(k1):
    """K1 = W2 W3 -> K = W0 W1."""
    w2, w3 = k1 >> 8, k1 & 0xFF
    w1 = w2 ^ w3
    w0 = w2 ^ g(w1, RCON1)
    return (w0 << 8) | w1


k1, k2 = key_pairs[0]
recovered_key = invert_key_expansion(k1)
print(f"recovered K = {recovered_key:04x}")
print("key_expansion(K) == (K1, K2):", key_expansion(recovered_key) == (k1, k2))
print("same as SECRET_KEY:", recovered_key == SECRET_KEY)

message = "ATTACK AT DAWN"
blocks = [int.from_bytes(message[i:i + 2].encode(), "big") for i in range(0, len(message), 2)]
intercepted = [encrypt(SECRET_KEY, b) for b in blocks]
recovered = b"".join(decrypt_round1(k1, decrypt_round2(k2, c)).to_bytes(2, "big")
                     for c in intercepted)
print("\nintercepted:", " ".join(f"{c:04x}" for c in intercepted))
print("decrypted  :", recovered.decode())

recovered K = 71d9
key_expansion(K) == (K1, K2): True
same as SECRET_KEY: True

intercepted: 2ecb 16a7 aa3e b911 975a d6a0 7c6d
decrypted  : ATTACK AT DAWN


### Task 3: Double mini cipher

The first 16 key bits go to the first mini cipher and the last 16 to the second. One mini cipher encryption or decryption counts as one operation.

In [14]:
def double_encrypt(key32, plaintext):
    return encrypt(key32 & 0xFFFF, encrypt(key32 >> 16, plaintext))


def double_mitm(pairs):
    """MITM on the double mini cipher. Returns (keys left, candidate counts, operations)."""
    P, C = pairs[0]
    table = {}
    for ka in range(1 << 16):
        table.setdefault(encrypt(ka, P), []).append(ka)
    candidates = []
    for kb in range(1 << 16):
        candidates += [(ka, kb) for ka in table.get(decrypt(kb, C), [])]
    ops, counts = 2 << 16, [len(candidates)]
    for P, C in pairs[1:]:
        ops += 2 * len(candidates)   # one encryption and one decryption per candidate
        candidates = [(ka, kb) for ka, kb in candidates if encrypt(ka, P) == decrypt(kb, C)]
        counts.append(len(candidates))
    return [(ka << 16) | kb for ka, kb in candidates], counts, ops

In [15]:
SECRET_KEY32 = 0x088AC4E1
double_pairs = [(p, double_encrypt(SECRET_KEY32, p)) for p in plaintexts]

start = time.perf_counter()
keys, counts, ops = double_mitm(double_pairs)
mitm_time = time.perf_counter() - start

print("candidates after each pair:", counts)
print(f"key found: {keys[0]:08x}, correct: {keys == [SECRET_KEY32]}")
print(f"operations: {ops:,} (2^{math.log2(ops):.2f}), time: {mitm_time:.1f} s")

candidates after each pair: [65506, 3, 1, 1, 1, 1, 1, 1, 1, 1]
key found: 088ac4e1, correct: True
operations: 262,104 (2^18.00), time: 1.3 s


Exhaustive search over 2^32 keys would take hours, so I timed 2^16 keys and scaled up.

In [16]:
P, C = double_pairs[0]
start = time.perf_counter()
fits = sum(double_encrypt(key32, P) == C for key32 in range(1 << 16))
per_key = (time.perf_counter() - start) / (1 << 16)
exhaustive_time = per_key * 2**32

print(f"timed 2^16 keys, {fits} of them fit the first pair")
print(f"all keys: 2^32 = {2**32:,}, operations: 2^33 = {2**33:,}")
print(f"estimated time: {exhaustive_time / 3600:.1f} hours")
print(f"the MITM was {exhaustive_time / mitm_time:,.0f} times faster")

timed 2^16 keys, 1 of them fit the first pair
all keys: 2^32 = 4,294,967,296, operations: 2^33 = 8,589,934,592
estimated time: 9.3 hours
the MITM was 26,159 times faster


### Task 4: Does the attack work on S-AES?

Same attack and key, but the pairs come from S-AES.

In [17]:
saes_pairs = [(p, saes_encrypt(SECRET_KEY, p)) for p in plaintexts]
_, counts = mitm_attack(saes_pairs)
print("candidates after each pair:", counts)

candidates after each pair: [65536, 0, 0, 0, 0, 0, 0, 0, 0, 0]


No key pair survives: with the first AddRoundKey the middle state depends on K, not only on K1.